# NeuroLab: Voice Processing and Emotion Analysis

This notebook demonstrates voice processing capabilities for emotion detection and mental state mapping.

**Features:**
- Audio preprocessing and feature extraction
- Emotion detection using TensorFlow models
- Rule-based fallback system
- Mental state mapping from emotions
- Multimodal analysis integration

**Author:** NeuroLab Team  
**License:** MIT

## 1. Setup and Imports

In [ ]:
# System imports
import sys
import os
sys.path.append('../')

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Audio processing
try:
    import librosa
    import soundfile as sf
    AUDIO_LIBS_AVAILABLE = True
    print("✓ Audio libraries available")
except ImportError:
    AUDIO_LIBS_AVAILABLE = False
    print("⚠️ Audio libraries not available. Install librosa and soundfile for full functionality.")

# NeuroLab modules
from src.utils.voice_processor import VoiceProcessor

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")

## 2. Voice Processor Initialization

In [ ]:
# Initialize voice processor
print("Initializing Voice Processor...")
voice_processor = VoiceProcessor()

# Display emotion to mental state mapping
print("\nEmotion to Mental State Mapping:")
print("=" * 40)
state_names = {0: 'Relaxed', 1: 'Focused', 2: 'Stressed'}
for emotion, state in voice_processor.emotion_to_state.items():
    print(f"{emotion.capitalize():>10} → {state_names[state]}")

print(f"\nProcessor Status:")
print(f"Device: {voice_processor.device}")
print(f"Sample Rate: {voice_processor.sample_rate} Hz")
print(f"Model Loaded: {voice_processor.model is not None}")

## 3. Audio Feature Extraction

In [ ]:
# Generate synthetic audio data for demonstration
def generate_synthetic_audio(emotion_type='neutral', duration=3.0, sample_rate=16000):
    """
    Generate synthetic audio data that mimics different emotional states
    """
    t = np.linspace(0, duration, int(sample_rate * duration))
    
    if emotion_type == 'angry':
        # High frequency, irregular pattern
        audio = 0.3 * np.sin(2 * np.pi * 800 * t) + 0.2 * np.sin(2 * np.pi * 1200 * t)
        audio += 0.1 * np.random.normal(0, 1, len(t))  # Add noise
    elif emotion_type == 'sad':
        # Low frequency, slow pattern
        audio = 0.4 * np.sin(2 * np.pi * 200 * t) + 0.2 * np.sin(2 * np.pi * 300 * t)
        audio *= np.exp(-t * 0.5)  # Decay
    elif emotion_type == 'happy':
        # Mid-high frequency, rhythmic pattern
        audio = 0.3 * np.sin(2 * np.pi * 440 * t) + 0.2 * np.sin(2 * np.pi * 660 * t)
        audio *= (1 + 0.3 * np.sin(2 * np.pi * 5 * t))  # Modulation
    else:  # neutral
        # Balanced frequency content
        audio = 0.2 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.sin(2 * np.pi * 880 * t)
    
    # Normalize
    audio = audio / np.max(np.abs(audio))
    return audio.astype(np.float32)

# Generate sample audio for different emotions
emotions = ['neutral', 'happy', 'sad', 'angry']
sample_audios = {}

print("Generating synthetic audio samples...")
for emotion in emotions:
    sample_audios[emotion] = generate_synthetic_audio(emotion)
    print(f"✓ Generated {emotion} audio: {len(sample_audios[emotion])} samples")

print("\n✓ Audio generation complete")

In [ ]:
# Visualize audio waveforms
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, (emotion, audio) in enumerate(sample_audios.items()):
    time = np.linspace(0, len(audio) / 16000, len(audio))
    axes[i].plot(time[:8000], audio[:8000], linewidth=1)  # Show first 0.5 seconds
    axes[i].set_title(f'{emotion.capitalize()} Audio Waveform', fontweight='bold')
    axes[i].set_xlabel('Time (s)')
    axes[i].set_ylabel('Amplitude')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Audio Feature Analysis

In [ ]:
# Extract basic audio features
def extract_basic_features(audio, sample_rate=16000):
    """
    Extract basic audio features for analysis
    """
    features = {}
    
    # RMS Energy
    features['rms_energy'] = np.sqrt(np.mean(audio ** 2))
    
    # Zero Crossing Rate
    zero_crossings = np.where(np.diff(np.signbit(audio)))[0]
    features['zero_crossing_rate'] = len(zero_crossings) / len(audio)
    
    # Spectral features using FFT
    fft = np.fft.fft(audio)
    magnitude = np.abs(fft[:len(fft)//2])
    freqs = np.fft.fftfreq(len(audio), 1/sample_rate)[:len(fft)//2]
    
    # Spectral centroid
    features['spectral_centroid'] = np.sum(freqs * magnitude) / np.sum(magnitude)
    
    # Spectral rolloff (95% of energy)
    cumsum = np.cumsum(magnitude)
    rolloff_idx = np.where(cumsum >= 0.95 * cumsum[-1])[0][0]
    features['spectral_rolloff'] = freqs[rolloff_idx]
    
    # Spectral bandwidth
    features['spectral_bandwidth'] = np.sqrt(np.sum(((freqs - features['spectral_centroid']) ** 2) * magnitude) / np.sum(magnitude))
    
    return features

# Extract features for all emotions
feature_data = []
for emotion, audio in sample_audios.items():
    features = extract_basic_features(audio)
    features['emotion'] = emotion
    feature_data.append(features)

# Create DataFrame
features_df = pd.DataFrame(feature_data)
print("Audio Features:")
print("=" * 50)
print(features_df.round(4))

In [ ]:
# Visualize feature distributions
feature_cols = ['rms_energy', 'zero_crossing_rate', 'spectral_centroid', 'spectral_rolloff', 'spectral_bandwidth']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i, feature in enumerate(feature_cols):
    if i < len(axes):
        bars = axes[i].bar(features_df['emotion'], features_df[feature], alpha=0.7)
        axes[i].set_title(f'{feature.replace("_", " ").title()}', fontweight='bold')
        axes[i].set_xlabel('Emotion')
        axes[i].set_ylabel('Value')
        axes[i].grid(True, alpha=0.3)
        
        # Color bars differently
        colors = ['blue', 'green', 'red', 'orange']
        for bar, color in zip(bars, colors):
            bar.set_color(color)

# Remove empty subplot
fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

## 5. Voice Processing Pipeline

In [ ]:
# Process audio samples through the voice processor
print("Processing Audio Samples...")
print("=" * 40)

results = []
for emotion, audio in sample_audios.items():
    # Convert to bytes (simulate file upload)
    audio_bytes = (audio * 32767).astype(np.int16).tobytes()
    
    try:
        # Process through voice processor
        result = voice_processor.process_audio(audio_bytes, sample_rate=16000)
        result['true_emotion'] = emotion
        results.append(result)
        
        print(f"\n{emotion.capitalize()} Audio:")
        print(f"  Detected Emotion: {result.get('emotion', 'unknown')}")
        print(f"  Confidence: {result.get('confidence', 0):.3f}")
        print(f"  Mental State: {result.get('mental_state', 'unknown')}")
        
    except Exception as e:
        print(f"Error processing {emotion} audio: {e}")
        results.append({
            'true_emotion': emotion,
            'emotion': 'error',
            'confidence': 0.0,
            'mental_state': 0
        })

print("\n✓ Audio processing complete")

In [ ]:
# Analyze processing results
results_df = pd.DataFrame(results)

if not results_df.empty and 'emotion' in results_df.columns:
    print("\nProcessing Results Summary:")
    print("=" * 30)
    print(results_df[['true_emotion', 'emotion', 'confidence', 'mental_state']].round(3))
    
    # Visualize results
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Confidence scores
    if 'confidence' in results_df.columns:
        bars1 = axes[0].bar(results_df['true_emotion'], results_df['confidence'], alpha=0.7)
        axes[0].set_title('Emotion Detection Confidence', fontweight='bold')
        axes[0].set_xlabel('True Emotion')
        axes[0].set_ylabel('Confidence Score')
        axes[0].grid(True, alpha=0.3)
        
        # Color bars
        colors = ['blue', 'green', 'red', 'orange']
        for bar, color in zip(bars1, colors[:len(bars1)]):
            bar.set_color(color)
    
    # Mental state mapping
    if 'mental_state' in results_df.columns:
        state_names = {0: 'Relaxed', 1: 'Focused', 2: 'Stressed'}
        state_labels = [state_names.get(state, f'State {state}') for state in results_df['mental_state']]
        
        bars2 = axes[1].bar(results_df['true_emotion'], results_df['mental_state'], alpha=0.7)
        axes[1].set_title('Mental State Classification', fontweight='bold')
        axes[1].set_xlabel('True Emotion')
        axes[1].set_ylabel('Mental State (0=Relaxed, 1=Focused, 2=Stressed)')
        axes[1].set_yticks([0, 1, 2])
        axes[1].set_yticklabels(['Relaxed', 'Focused', 'Stressed'])
        axes[1].grid(True, alpha=0.3)
        
        # Color bars
        for bar, color in zip(bars2, colors[:len(bars2)]):
            bar.set_color(color)
    
    plt.tight_layout()
    plt.show()
else:
    print("No valid results to display")

## 6. Multimodal Integration Example

In [ ]:
# Simulate multimodal analysis (EEG + Voice)
def simulate_multimodal_analysis():
    """
    Simulate combining EEG and voice data for comprehensive mental state assessment
    """
    # Simulate EEG data for different states
    eeg_states = {
        'relaxed': {'alpha': 25, 'beta': 8, 'theta': 10, 'delta': 5, 'gamma': 3},
        'focused': {'alpha': 15, 'beta': 25, 'theta': 5, 'delta': 3, 'gamma': 8},
        'stressed': {'alpha': 8, 'beta': 35, 'theta': 12, 'delta': 6, 'gamma': 20}
    }
    
    # Simulate voice emotions for each state
    voice_emotions = {
        'relaxed': {'emotion': 'calm', 'confidence': 0.85, 'mental_state': 0},
        'focused': {'emotion': 'happy', 'confidence': 0.78, 'mental_state': 1},
        'stressed': {'emotion': 'angry', 'confidence': 0.92, 'mental_state': 2}
    }
    
    multimodal_results = []
    
    for state in ['relaxed', 'focused', 'stressed']:
        eeg_data = eeg_states[state]
        voice_data = voice_emotions[state]
        
        # Simple fusion: weighted average
        eeg_weight = 0.6
        voice_weight = 0.4
        
        # Map state names to numbers
        state_mapping = {'relaxed': 0, 'focused': 1, 'stressed': 2}
        eeg_state = state_mapping[state]
        voice_state = voice_data['mental_state']
        
        # Weighted fusion
        fused_confidence = (eeg_weight * 0.9) + (voice_weight * voice_data['confidence'])
        fused_state = int(round((eeg_weight * eeg_state) + (voice_weight * voice_state)))
        
        result = {
            'true_state': state,
            'eeg_state': eeg_state,
            'voice_emotion': voice_data['emotion'],
            'voice_state': voice_state,
            'fused_state': fused_state,
            'fused_confidence': fused_confidence,
            'eeg_features': eeg_data
        }
        
        multimodal_results.append(result)
    
    return multimodal_results

# Run multimodal simulation
multimodal_data = simulate_multimodal_analysis()

print("Multimodal Analysis Results:")
print("=" * 50)
for result in multimodal_data:
    print(f"\nTrue State: {result['true_state'].capitalize()}")
    print(f"  EEG State: {result['eeg_state']}")
    print(f"  Voice Emotion: {result['voice_emotion']}")
    print(f"  Voice State: {result['voice_state']}")
    print(f"  Fused State: {result['fused_state']}")
    print(f"  Fused Confidence: {result['fused_confidence']:.3f}")

In [ ]:
# Visualize multimodal fusion results
multimodal_df = pd.DataFrame(multimodal_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# State comparison
states = multimodal_df['true_state']
x_pos = np.arange(len(states))

axes[0].bar(x_pos - 0.2, multimodal_df['eeg_state'], 0.4, label='EEG State', alpha=0.7)
axes[0].bar(x_pos + 0.2, multimodal_df['voice_state'], 0.4, label='Voice State', alpha=0.7)
axes[0].set_xlabel('True State')
axes[0].set_ylabel('Predicted State')
axes[0].set_title('EEG vs Voice State Prediction', fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([s.capitalize() for s in states])
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Fusion results
axes[1].bar(states, multimodal_df['fused_state'], alpha=0.7, color='purple')
axes[1].set_xlabel('True State')
axes[1].set_ylabel('Fused State')
axes[1].set_title('Multimodal Fusion Results', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Confidence scores
axes[2].bar(states, multimodal_df['fused_confidence'], alpha=0.7, color='green')
axes[2].set_xlabel('True State')
axes[2].set_ylabel('Confidence Score')
axes[2].set_title('Fusion Confidence', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Performance Analysis

In [ ]:
# Analyze the effectiveness of multimodal fusion
print("Multimodal Fusion Analysis:")
print("=" * 40)

# Calculate accuracy for each modality
state_mapping = {'relaxed': 0, 'focused': 1, 'stressed': 2}
true_states = [state_mapping[result['true_state']] for result in multimodal_data]
eeg_states = [result['eeg_state'] for result in multimodal_data]
voice_states = [result['voice_state'] for result in multimodal_data]
fused_states = [result['fused_state'] for result in multimodal_data]

eeg_accuracy = sum(1 for t, p in zip(true_states, eeg_states) if t == p) / len(true_states)
voice_accuracy = sum(1 for t, p in zip(true_states, voice_states) if t == p) / len(true_states)
fusion_accuracy = sum(1 for t, p in zip(true_states, fused_states) if t == p) / len(true_states)

print(f"EEG-only Accuracy: {eeg_accuracy:.3f}")
print(f"Voice-only Accuracy: {voice_accuracy:.3f}")
print(f"Multimodal Fusion Accuracy: {fusion_accuracy:.3f}")

# Calculate average confidence
avg_confidence = np.mean([result['fused_confidence'] for result in multimodal_data])
print(f"Average Fusion Confidence: {avg_confidence:.3f}")

# Visualize accuracy comparison
plt.figure(figsize=(10, 6))
methods = ['EEG Only', 'Voice Only', 'Multimodal Fusion']
accuracies = [eeg_accuracy, voice_accuracy, fusion_accuracy]
colors = ['blue', 'orange', 'green']

bars = plt.bar(methods, accuracies, color=colors, alpha=0.7)
plt.title('Modality Comparison: Classification Accuracy', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy')
plt.ylim(0, 1.1)
plt.grid(True, alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Voice processing analysis complete!")